# Fine-Tuning and Cleaning

This notebook fine-tunes an open-access LLM from [HuggingFace](https://huggingface.co) to correct OCR errors in historical newspaper text. It is the optional third stage of the THISTLE pipeline and is most useful if the baseline results from `post_ocr_llm.ipynb` are not satisfactory.

## What is fine-tuning and why do it?

The baseline test in `post_ocr_llm.ipynb` evaluates how well a general-purpose LLM corrects OCR errors without any task-specific training. Fine-tuning adapts the model further by training it on examples from your own dataset — in this case, pairs of raw OCR text and manually transcribed ground truth. The goal is to teach the model the specific error patterns and language conventions of your corpus.

For *The Scotsman* data, fine-tuning is particularly motivated by the models' tendencies to modernise or change historical language, generate plausible but incorrect text when the OCR input is very degraded, and add information not present in the original text. Successful fine-tuning aims to teach the model to avoid these behaviours.

## LoRA fine-tuning

Full fine-tuning of LLMs requires updating billions of parameters and can require significant GPU resources to be impractical, even for HPC users who may face long queues. This notebook uses the LoRA (Low-Rank Adaptation) fine tuning approach to instead trains a small number of additional parameters while retaining the original model weights. For more background on LoRA, see `code/README.md`.

**Run cells from top to bottom.**


## For working in Colab

## Connect to the GPU:
In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'. **NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

## Connect your Google Drive:
Run the code block below to mount your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install and import dependencies
Fine-tuning requires several additional libraries. Run this cell first.

**Note:** A GPU is required. In Google Colab, enable GPU via **Runtime > Change runtime type > T4 GPU**. For HPC users, you will need to request a GPU node in your SLURM job script — see `HPC/README.md` for more information about how to do this.

**Also note:** Downloading some HuggingFace models requires you to register for an account, accept the model's licence terms, and generate an access token which you will need to insert in this notebook. Check the HuggingFace model card to see if this is the case for your selection. If so:

1. Create a free account at [huggingface.co](https://huggingface.co)
2. Navigate to the model page (e.g. [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)) and accept the licence
3. Create an access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) **with Read permissions**
4. Take note of your token; you will need to paste it below

**Also also note:** It is best practice to keep API keys and credentials, such as your HuggingFace access token, secret. If you will be reusing, sharing, or uploading this notebook, you may wish to consider storing your access token in an environment variable (`.env`. file). Read more about how to do this on StackOverflow here: https://stackoverflow.com/questions/56995350/best-practices-python-where-to-store-api-keys-tokens

In [ ]:
!pip install pandas transformers torch peft trl bitsandbytes accelerate datasets tqdm huggingface_hub

In [ ]:
# run if you are using a gated model (e.g. Llama 3); otherwise, you can skip this cell
from huggingface_hub import login

hf_token = 'your_huggingface_token_here'  # paste your HuggingFace access token here. Note: keep your access token secret! Remove it from this notebook before sharing or uploading.
login(token=hf_token)

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import torch
from datasets import Dataset

## 2. Load and split data

This cell loads the post-OCR output CSV produced in the previous step and splits it into training and testing sets. The training set is used to fine-tune the model; the test set is held out for evaluation.

**Note on split size:** `test_size=0.2` reserves 20% of the data for testing. With a dataset of 100 articles (as in the THISTLE ground truth), this translates to an 80/20 training/testing split. You may wish to adjust `test_size`.

In [ ]:
# paste the path to the post_ocr_output.csv saved in the previous step (post_ocr_llm.ipynb)
# example (Colab): "/content/drive/My Drive/your_project_folder/post_ocr_output.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/post_ocr_output.csv"
df = pd.read_csv('insert_your_path_to/post_ocr_output.csv')

# split into training (80%) and test (20%) sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=13)

## 3. Prepare dataset for fine-tuning

Each training example is formatted as an instruction–response pair:

- **Input (prompt):** the OCR correction instruction with the raw ProQuest OCR text inserted
- **Output (completion):** the manually transcribed ground truth

The model learns to map from the (prompt + raw OCR text) to the corrected transcription. This is the same task as the baseline prompt test, but the model is now shown many examples of what a correct output looks like for *this specific corpus*, which helps it learn to avoid common errors such as modernising archaic language.

In [ ]:
def format_instruction(row):
    return {
        # 'ocr' is the raw OCR text column; 'ground_truth' is the manually transcribed column
        # update these column names if your dataframe uses different names
        "prompt": f"please correct the OCR without modernising or changing the language and without adding new information {row['ocr']}",
        "completion": row['ground_truth']
    }

train_dataset = Dataset.from_pandas(train_df).map(format_instruction)

## 4. Fine-tuning setup

Set the model and training parameters below.

### Choosing a base model
Set `model_id` to the full HuggingFace model identifier for the model you want to fine-tune. The model used in this project was `meta-llama/Meta-Llama-3-8B-Instruct`. To browse alternatives, visit [huggingface.co/models](https://huggingface.co/models) and filter by 'Text Generation'.

### 4-bit quantisation (QLoRA)
To reduce GPU memory usage, the model is loaded in 4-bit quantisation using `bitsandbytes`. This is enabled by default. If you prefer to fine-tune without quantisation (requires more GPU memory), you can remove the `quantization_config` argument from `AutoModelForCausalLM.from_pretrained`.

### LoRA parameters
- **`r`**: the rank of the LoRA adapters (lower = fewer parameters, faster training; higher = more capacity)
- **`lora_alpha`**: scaling factor; typically set to `2 * r`
- **`target_modules`**: which layers to apply LoRA to; `q_proj` and `v_proj` are standard choices for transformer models
- **`lora_dropout`**: dropout rate for regularisation

In [ ]:
# paste the full HuggingFace model ID here
# example: "meta-llama/Meta-Llama-3-8B-Instruct"
model_id = "insert_huggingface_model_id"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

## 5. Train

This cell runs the fine-tuning. Training time depends on the size of the model and dataset, and the GPU available. On a Colab T4 GPU with the THISTLE dataset (80 training examples) and Llama 3 8B, expect approximately 30–60 minutes for 3 epochs.

Checkpoints are saved every 100 steps to the `output_dir`. If training is interrupted, you can resume from the most recent checkpoint.

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama3-ocr-ft",    # directory to save checkpoints
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_steps=100,
    logging_steps=10,
    fp16=True
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    dataset_text_field="prompt",
    max_seq_length=1024,
    packing=False
)

trainer.train()

## 6. Deploy on test set

This cell generates corrected text for each row in the held-out test set using the fine-tuned model. The predictions are saved to a new column (`ft_prediction`) in the test dataframe, which is then saved to a CSV for evaluation in `accuracy_and_results.ipynb`.

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def generate_prediction(text):
    prompt = f"please correct the OCR without modernising or changing the language and without adding new information {text}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return prediction

# apply to the 'original_ocr' column in the test set
# update the column name if your dataframe uses a different name
test_df['ft_prediction'] = test_df['original_ocr'].progress_apply(generate_prediction)


## 7. Save results

Save the test set with fine-tuned predictions to a CSV for evaluation in `accuracy_and_results.ipynb`.

In [ ]:
# paste the path where you would like to save the fine-tuned output CSV
# example (Colab): "/content/drive/My Drive/your_project_folder/llama_fine_tuned_output.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/llama_fine_tuned_output.csv"
test_df.to_csv('insert_your_output_path/llama_fine_tuned_output.csv', index=False)
print('Saved successfully.')